# actas-cnn — entrenamiento portable (Colab/Kaggle, GPU)

Notebook para entrenar **ResNet-18 estilo CIFAR** sobre el dataset
empaquetado de actas Presidenciales ONPE en una GPU gratuita (T4 en
Colab, T4×2 en Kaggle). Soporta ablations (label smoothing,
RandAugment, mixup, cosine LR).

## Cómo correr

1. **Abrir en Colab**: `Runtime` → `Change runtime type` →
   **T4 GPU**.
2. **`Runtime` → `Run all`**. No requiere tokens ni configuración
   manual. El dataset se baja desde el HF public dataset repo
   `f3r21/actas-cnn-dataset`.
3. Al terminar, el navegador baja **automáticamente** dos archivos:
   - `resnet18_<SUFFIX>_best.pt` — el checkpoint del modelo.
   - `evaluate_val_<SUFFIX>.csv` — métricas por field.

   Movelos a tu repo local en `checkpoints/` y `data/`
   respectivamente.

Detalle adicional en `docs/08-setup-colab.md`.

In [ ]:
# 1. Clonar repo + cwd
import os, sys

REPO_URL = "https://github.com/f3r21/actas-cnn.git"
REPO_DIR = "actas-cnn"

if not os.path.isdir(REPO_DIR):
    os.system(f"git clone {REPO_URL}")
os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())
print("cwd:", os.getcwd())

In [ ]:
# 2. Instalar dependencias (torch ya viene en Colab/Kaggle con CUDA)
os.system("pip install -q -r requirements.txt")

In [ ]:
# 4. Descargar bundle (dataset publico, sin token) y descomprimir
from huggingface_hub import hf_hub_download
from config import REMOTE
from pathlib import Path

bundle_local = hf_hub_download(
    repo_id=REMOTE.hf_dataset_repo,
    repo_type="dataset",
    filename="data_bundle.tar.gz",
)
print(f"bundle bajado: {bundle_local}")

# Descomprimir en la raiz del repo (extrae a data/, templates.json, fiducial_anchors.json)
os.system(f"tar -xzf {bundle_local} -C .")
print("descomprimido. Estructura:")
os.system("ls -lh data/ | head; echo '---'; ls data/crops_train/ | head -3")

In [ ]:
# 5. Entrenar (ablation principal)
ARCH = "resnet18"
EPOCHS = 40   # mas epochs aprovechando GPU; cosine LR se ajusta automatico
LABEL_SMOOTHING = 0.1
RANDAUGMENT = True
MIXUP = 0.2
COSINE_LR = True
SUFFIX = "colab_ls_ra_mu_cos_e40"

flags = []
if LABEL_SMOOTHING > 0:
    flags.append(f"--label-smoothing {LABEL_SMOOTHING}")
if RANDAUGMENT:
    flags.append("--randaugment")
if MIXUP > 0:
    flags.append(f"--mixup {MIXUP}")
if COSINE_LR:
    flags.append("--cosine-lr")
flags.append(f"--suffix {SUFFIX}")

cmd = (f"python train.py --manifest data/manifest_train.csv "
       f"--root data/crops_train --arch {ARCH} --epochs {EPOCHS} {' '.join(flags)}")
print(cmd)
os.system(cmd)

In [ ]:
# 6. Evaluar (digit / field / acta-level + reconstruccion totales)
ckpt = f"checkpoints/{ARCH}_{SUFFIX}_best.pt"
os.system(f"python scripts/evaluate.py --split val --checkpoint {ckpt} "
          f"--out-csv data/evaluate_val_{SUFFIX}.csv")

In [ ]:
# 7. Descargar resultados al navegador (sin tokens)
# Baja el checkpoint .pt y el CSV de evaluate a tu Downloads local.
# Despues movelos al repo local: checkpoints/ y data/.
try:
    from google.colab import files
    files.download(ckpt)
    files.download(f"data/evaluate_val_{SUFFIX}.csv")
    print(f"descargados al navegador: {Path(ckpt).name} + evaluate CSV")
except ImportError:
    # En Kaggle / otro entorno sin google.colab, los archivos quedan en
    # el VM. Bajalos desde el panel lateral.
    print(f"archivos en VM (bajar manualmente):")
    print(f"  {ckpt}")
    print(f"  data/evaluate_val_{SUFFIX}.csv")

In [ ]:
os.system("python train.py --manifest data/manifest.csv --root data/crops --arch deep --epochs 20 --push")